# Breast Cancer Classification — Training and Evaluation

Run this notebook in **Google Colab or Jupyter**, not VS Code, to satisfy the model-training step.

Dataset: Wisconsin Diagnostic Breast Cancer, provided by scikit-learn. It is commonly mirrored on Kaggle. If your instructor requires the CSV to be downloaded from Kaggle specifically, upload the required CSV and adjust the data-loading cell.

**Educational demo only — not for clinical use.**


In [ ]:
%pip install -q scikit-learn pandas joblib


In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
import joblib, json
from pathlib import Path


In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

print("Rows:", len(X))
print("Features:", X.shape[1])
print("Class mapping:", {0: "malignant", 1: "benign"})
display(X.head())


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=5000, random_state=42)),
])
model.fit(X_train, y_train)

predictions = model.predict(X_test)
probabilities = model.predict_proba(X_test)[:, 1]
metrics = {
    "accuracy": round(accuracy_score(y_test, predictions), 4),
    "precision_weighted": round(precision_score(y_test, predictions, average="weighted"), 4),
    "recall_weighted": round(recall_score(y_test, predictions, average="weighted"), 4),
    "f1_weighted": round(f1_score(y_test, predictions, average="weighted"), 4),
    "roc_auc_benign_class": round(roc_auc_score(y_test, probabilities), 4),
}
metrics


In [ ]:
print(classification_report(y_test, predictions, target_names=dataset.target_names))


## Save model and metadata

The FastAPI service uses `model.joblib`. The optional `model.pkl` is included for the assignment's `.pkl` requirement.


In [ ]:
artifact = {
    "model": model,
    "feature_names": list(X.columns),
    "target_names": list(dataset.target_names),
    "class_mapping": {"0": "malignant", "1": "benign"},
    "metrics": metrics,
}
joblib.dump(artifact, "model.joblib")
import pickle
with open("model.pkl", "wb") as f:
    pickle.dump(artifact, f)
print("Saved model.joblib and model.pkl")


In [ ]:
# Download from Colab if needed
from google.colab import files
files.download("model.joblib")
files.download("model.pkl")
